In [ ]:
from langgraph.graph import StateGraph, START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import MemorySaver

In [ ]:
from langgraph.graph.message import add_messages

class ChatState(TypedDict):
  messages: Annotated[list[BaseMessage], add_messages]

In [ ]:
llm = ChatOpenAI()


def chat_node(state: ChatState):
  # take user query from state
  messages = state['messages']

  # send to llm
  response = llm.invoke(messages)

  # response store state
  return {'messages': [response]}

In [ ]:
checkpointer = MemorySaver()
graph = StateGraph(ChatState)

graph.add_node('chat_node', chat_node)

graph.add_edge(START, 'chat_node')
graph.add_edge('chat_node', END)

chatbot = graph.compile(checkpointer=checkpointer)

In [ ]:
thread_id = '1'
while True:
  user_message = input('Type here: ')
  print('User:', user_message)

  if user_message.strip().lower() in ['exit', 'quit', 'bye']:
    break

  config = { 'configurable': {'thread_id': thread_id} }
  for message_chunk, metadata in chatbot.stream(
    {'messages': [HumanMessage(content=user_message)]}, 
    config=config,
    stream_mode="messages"
  ):
    if message_chunk.content:
      print(message_chunk.content, end="|", flush=True)

  